In [1]:
import random
import json

#### JSON from the editor
```json
{
  "nodes": [
    {
      "width": 240,
      "height": 276,
      "id": "0.05272377425322505-1756500835216",
      "type": "eventType",
      "position": {
        "x": 145.53354881065553,
        "y": 1196.7836919286149
      },
      "data": {
        "box": {
          "newEventVars": {
            "0": [
              "payment reminder"
            ],
            "1": [
              "pay order"
            ]
          },
          "newObjectVars": {
            "0": [
              "orders"
            ]
          },
          "filters": [
            {
              "type": "O2E",
              "object": 0,
              "event": 0,
              "qualifier": null
            },
            {
              "type": "O2E",
              "object": 0,
              "event": 1,
              "qualifier": null
            }
          ],
          "sizeFilters": [],
          "constraints": [
            {
              "type": "Filter",
              "filter": {
                "type": "TimeBetweenEvents",
                "from_event": 0,
                "to_event": 1,
                "min_seconds": 0,
                "max_seconds": null
              }
            }
          ],
          "evVarLabels": {},
          "obVarLabels": {}
        }
      },
      "positionAbsolute": {
        "x": 145.53354881065553,
        "y": 1196.7836919286149
      },
      "selected": true,
      "dragging": false
    }
  ],
  "edges": []
}
```

#### JSON given to the DB
```json
{
  "tree": {
    "nodes": [
      {
         "Box": [
           {
             "newEventVars": { "0": ["payment reminder"], "1": ["pay order"] },
             "newObjectVars": { "0": ["orders"] },
             "filters": [
               { "type": "O2E", "object": 0, "event": 0, "qualifier": null },
               { "type": "O2E", "object": 0, "event": 1, "qualifier": null }
             ],
             "sizeFilters": [],
             "constraints": [
               {
                 "type": "Filter",
                 "filter": { "type": "TimeBetweenEvents", "from_event": 0, "to_event": 1, "min_seconds": 0, "max_seconds": null }
               }
             ],
             "evVarLabels": {},
             "obVarLabels": {}
           },
           [] // IDs of any connected child nodes in the tree
         ]
      }
    ],
    "edgeNames": []
  },
  "database": "DuckDB", // Target Database Engine (if doing DB translation)
  "table_mappings": { ... }
}
```

In [ ]:
patterns = [
    "EVENT_A happens before EVENT_B",
    "Whenever EVENT_A occurs, EVENT_B eventually occurs",
    "After EVENT_A, EVENT_B must not occur",
    "EVENT_A and EVENT_B refer to the same OBJECT",
    "EVENT_B happens within TIME hours after EVENT_A"
]

domains = {
    "e-commerce": {
        "events": ["create order", "add item", "pay order", "ship package", "cancel order"],
        "objects": ["customers", "employees", "items", "orders", "packages", "products"]
    },
    "recruitment": {
        "events": ["submit application", "review application", "approve application", "reject application"],
        "objects": ["applications", "candidates"]
    },
    "software_dev": {
        "events": ["open issue", "submit pull request", "review pull request", "merge pull request", "close issue"],
        "objects": ["issues", "pull requests", "commits"]
    },
    "distribution": {
        "events": ["Apply Discount", "Apply Payment", "Approve Order", "Approve Return", "Generate Invoice", 
                   "Generate Pick List", "Generate Shipping Label", "Handle Payment Discrepancy", 
                   "Issue Credit Note", "Issue Payment Reminder", "Pack Items", "Perform Credit Check", 
                   "Pick Items", "Process Return Request", "Receive Customer Order", "Receive Payment", 
                   "Reject Order", "Schedule Order Fulfillment", "Send Invoice to Customer", "Ship Order", 
                   "Update Inventory", "Validate Customer Order"],
        "objects": ["CreditNote", "Customer", "Dispute", "Invoice", "Order", "Payment", "Product", "Return", "Shipment"]
    },
    "procurement": {
        "events": ["Approve PO", "Approve PR", "Create PO", "Create PO (Batch)", "Create PR", "Make Payment", 
                   "Receive Goods", "Receive Invoice", "Reject PR", "Send PO to Supplier", 
                   "Three-way Match - Mismatch Found", "Three-way Match - Successful"],
        "objects": ["Goods Receipt", "Purchase Requisition", "Payment", "Invoice", "Purchase Order", "Supplier"]
    },
    "02_p2p": {
        "events": ["Approve PO", "Approve PR", "Create PO", "Create PO (Batch)", "Create PR", "Make Payment", 
                   "Receive Goods", "Receive Invoice", "Reject PR", "Send PO to Supplier", "Three-way Match - Mismatch Found", 
                   "Three-way Match - Successful"],
        "objects": ["Goods Receipt", "Invoice", "Payment", "Purchase Order", "Purchase Requisition", "Supplier"]
    },
    "hiring": {
        "events": ['Approve Job Requisition', 'Attend Orientation Training', 'Candidate Accept Offer', 'Candidate Reject Offer', 
                   'Complete Assessment', 'Complete Background Check', 'Complete HR Paperwork', 'Complete Onboarding', 
                   'Conduct Initial Interview', 'Conduct Panel Interview', 'Create Job Requisition', 'Evaluate Interview', 
                   'Initiate Background Check', 'Post Job Opening', 'Prepare Offer Letter', 'Review Assessment', 
                   'Schedule Initial Interview', 'Schedule Panel Interview', 'Screen Resume', 'Send Assessment', 
                   'Send Offer Letter', 'Send Rejection Letter', 'Start Onboarding', 'Submit Application'],
        "objects": ['Interviewer', 'HiringManager', 'Recruiter', 'Assessment', 'Candidate', 'JobRequisition']
    },
    # not sure about the events and objects here, but can be added later
    # "angular_github_commits": {
    #     "events": ["add", "address", "aio", "benchmark", "bug", "build", "bump", "check", "chore", "ci", "cleanup", "code", "condition", "configure", "consolidated", "copy", "design", "developer.md", "docs", "don't", "edit", "example", "examples", "feat", "filetree", "finished", "first", "fix", "form", "format", "include", "initial", "keep", "make", "merge", "misc", "more", "moved", "patch", "perf", "produce", "read", "readme", "refactor", "release", "remove", "rendererv2", "repackaging", "replaced", "resolve", "revert", "roll", "router", "squash", "squash-me", "style", "table", "temporary", "test", "text", "tools", "transpiler", "tweak", "typo", "update", "use", "wip"],
    #     "objects": ["branches", "files"]
    # },
    "hospital": {
        "events": ['Appointment Scheduling', 'Assign Bed', 'Diagnostic Test Order', 'Initial Evaluation', 
                   'Insurance Verification', 'Patient Arrival', 'Perform Diagnostic Test', 'Pre-Admission Consultation', 
                   'Registration'],
        "objects": ['Patient', 'InsuranceCompany', 'Physician', 'LabTechnician', 'Nurse', 'DiagnosticEquipment', 'Room', 'Bed']
    },
    "container_logistics": {
        "events": ['Book Vehicles', 'Bring to Loading Bay', 'Collect Goods', 'Create Transport Document', 'Depart', 
                   'Drive to Terminal', 'Load Truck', 'Load to Vehicle', 'Order Empty Containers', 'Pick Up Empty Container', 
                   'Place in Stock', 'Register Customer Order', 'Reschedule Container', 'Weigh'],
        "objects": ['Container', 'Customer Order', 'Forklift', 'Handling Unit', 'Transport Document', 'Truck', 'Vehicle']
    },
    "lending": {
        "events": ['Application Rejected', 'Appointment Denied', 'Hand In Credit Appliaction', 'Loan Denied', 
                   'Make Visit to Assess Colatteral', 'Request Appointment', 'Request Co-Signer On Loan', 
                   'Set Appointment', 'Sign Loan Agreement', 'Submit File to Underwriter', 'Verify Borrowers Information', 
                   'skipped_examination'],
        "objects": ['Person']
    },
    # no. just no...
    # "Period20_procedure_steps_all_additional_event_types": {
    #     "events": ["1. Beratung", "1. Beratung (Einzelpl\u00e4ne)", "1. Beratung (Gesetzentwurf)", "1. Durchgang", 
    #                "2. Beratung", "2. Beratung (Einzelpl\u00e4ne)", "2. Beratung (Gesetzentwurf)", 
    #                "2. Beratung und Schlussabstimmung", "2. Durchgang", "3. Beratung", "Absetzung von der TO", 
    #                "Abstimmung", "Abstimmung \u00fcber Vermittlungsvorschlag", "Aktuelle Stunde", 
    #                "Allgemeine Verwaltungsvorschrift", "Amtliche Mitteilung: R\u00fccknahme", "Amts\u00fcbernahme", 
    #                "Ansprache", "Antrag", "Antrag auf Anrufung des Vermittlungsausschusses", 
    #                "Antrag auf Einsetzung einer Enquete-Kommission", "Antrag auf Einsetzung eines Untersuchungsausschusses", 
    #                "Antrag auf Entlastung der BRg", "Antrag auf Entschlie\u00dfung", "Antrag auf erneute Behandlung", 
    #                "Antrag auf erneute \u00dcberweisung", "Antrag des Pr\u00e4sidenten des BRH", 
    #                "Antrag zur Arbeit der Enquete-Kommission", "Antrag zur Gesch\u00e4ftsordnung", 
    #                "Antrag zur Weitergeltung von Gesch\u00e4ftsordnungsrecht", "Antwort", "Ausfertigung", 
    #                "Ausschusszuweisung", "Ausschuss\u00fcberweisung", "Aussprache", "BR-Sitzung", 
    #                "Bekanntgabe der Bildung der Bundesregierung und Eidesleistung der Bundesminister", 
    #                "Bekanntgabe gem\u00e4\u00df \u00a7 23 Abs. 1 Gesch\u00e4ftsordnung BR", 
    #                "Bemerkungen des Bundesrechnungshofes", "Beratung", "Beratung des Antrags auf erneute Behandlung", 
    #                "Beratung des Berichts \u00fcber den Stand der Ausschussberatung", 
    #                "Beratung \u00fcber Antrag auf erneute Behandlung", 
    #                "Beratung \u00fcber Bericht gem\u00e4\u00df \u00a7 62 Abs. 2 Gesch\u00e4ftsordnung BT", "Bericht", 
    #                "Bericht der Kommission", "Bericht gem\u00e4\u00df \u00a7 96 Gesch\u00e4ftsordnung BT", 
    #                "Bericht \u00fcber den Stand der Ausschussberatung gem\u00e4\u00df \u00a7 62 Abs. 2 Gesch\u00e4ftsordnung BT", 
    #                "Berichtigung", "Berichtigung zum Beschluss", "Berichtigung zum Gesetzesbeschluss", "Beschlussdrucksache", 
    #                "Beschlussdrucksache / Verordnungsentwurf", "Beschlussempfehlung", 
    #                "Beschlussempfehlung des Wahlpr\u00fcfungsausschusses", "Beschlussempfehlung und Bericht", 
    #                "Beschlussempfehlung und Bericht betreffend Wahleinspr\u00fcche", 
    #                "Beschlussempfehlung und Bericht des Untersuchungsausschusses", "Durchgang", "Einigungsvorschlag", 
    #                "Einleitende Ausf\u00fchrungen und Befragung der Bundesregierung", "Empfehlungen der Aussch\u00fcsse", 
    #                "Erg\u00e4nzende Antwort", "Erg\u00e4nzende schriftliche Antwort", "Erg\u00e4nzung zum Gesetzentwurf", 
    #                "Erkl\u00e4rung", "Erkl\u00e4rung der BRg", 
    #                "Erneute \u00dcberweisung gem\u00e4\u00df \u00a7 93 Gesch\u00e4ftsordnung BT", "Feststellung der Tagesordnung", 
    #                "Fortsetzung: 1. Beratung", "Fortsetzung: 2. Beratung", "Fortsetzung: Abstimmung", "Fortsetzung: Aussprache", 
    #                "Fortsetzung: Beratung", "Frage zum Grund der Frist\u00fcberschreitung", 
    #                "Gegen\u00e4u\u00dferung der BRg zur Stellungnahme des BR", 
    #                "Gesch\u00e4ftsordnungsantrag auf Absetzung der 1. Beratung von der TO", 
    #                "Gesch\u00e4ftsordnungsantrag auf Absetzung von der TO", "Gesch\u00e4ftsordnungsantrag auf Aufsetzung auf die TO", 
    #                "Gesch\u00e4ftsordnungsantrag auf Aufsetzung der 1. Beratung auf die TO", "Gesch\u00e4ftsordnungsantrag zur Beratung", 
    #                "Gesetzentwurf", "Gesetzesantrag", "Gro\u00dfe Anfrage", "Inkrafttreten", "Kleine Anfrage", "L\u00e4nderantrag", 
    #                "Mitteilung", "Mitteilung des federf\u00fchrenden Ausschusses", "Mitteilung zur Subsidiarit\u00e4tsfrist", 
    #                "M\u00fcndliche Frage", "M\u00fcndliche Frage/M\u00fcndliche Antwort", "M\u00fcndliche Frage/Schriftliche Antwort", 
    #                "Nachtrag: 2. Beratung", "Nachtrag: 2. Beratung und Schlussabstimmung", "Nachtrag: 3. Beratung", 
    #                "Nachtrag: Abstimmung \u00fcber Vermittlungsvorschlag", "Nachtrag: Antrag zur Gesch\u00e4ftsordnung", 
    #                "Nachtrag: Beratung", "Nachtrag: M\u00fcndliche Frage/Schriftliche Antwort", 
    #                "Nachtrag: Regierungserkl\u00e4rung und Aussprache", "Nachtr\u00e4gliche \u00dcberweisung", 
    #                "Nachtr\u00e4gliche \u00dcberweisung gem\u00e4\u00df \u00a7 80 Gesch\u00e4ftsordnung BT", 
    #                "Neuabdruck der Antwort", "Nichtbeantwortung", "Ordnungsruf/Ordnungsgeld/Sitzungsausschluss", 
    #                "Plenarantrag", "Protokollerkl\u00e4rung/Begleiterkl\u00e4rung zum Vermittlungsverfahren", 
    #                "Regierungserkl\u00e4rung und Aussprache", "R\u00fccknahme", "R\u00fccknahme der Verordnung", 
    #                "R\u00fccknahme der Vorlage", "Schriftliche Frage zur m\u00fcndlichen Beantwortung wegen Frist\u00fcberschreitung", 
    #                "Schriftliche Frage/Schriftliche Antwort", "Unterrichtung", "Unterrichtung (Stellungnahme)", 
    #                "Unterrichtung durch den Vermittlungsausschuss", 
    #                "Unterrichtung \u00fcber Anrufung des Vermittlungsausschusses durch den BR", 
    #                "Unterrichtung \u00fcber Anrufung des Vermittlungsausschusses durch die BRg", 
    #                "Unterrichtung \u00fcber Beschluss des BT", "Unterrichtung \u00fcber Gesetzesbeschluss des BT", 
    #                "Unterrichtung \u00fcber Stellungnahme der BRg", "Unterrichtung \u00fcber Stellungnahme des BR", 
    #                "Unterrichtung \u00fcber Stellungnahme des BR und Gegen\u00e4u\u00dferung der BRg", 
    #                "Unterrichtung \u00fcber Zustimmungsversagung durch den BR", "Unterrichtung: Keine Ausschuss\u00fcberweisung", 
    #                "Vereidigung", "Verk\u00fcndung", "Vermittlungsvorschlag", "Verordnung", "Verordnungsantrag", 
    #                "Vertrauensantrag nach Art. 68 Grundgesetz", "Vorschlag", "Vorschlag / Vorlage / Mitteilung", "Wahl", 
    #                "Wahl des Bundespr\u00e4sidenten durch die Bundesversammlung", "Wahl des Pr\u00e4sidiums", 
    #                "Wahl des Vorsitzenden der Europakammer", "Wahlvorschlag", 
    #                "Zugeh\u00f6riger Vorgang: Antrag zur Mitwirkung des BT in EU-Angelegenheiten", 
    #                "Zur\u00fcckverweisung an die Aussch\u00fcsse", "Zur\u00fcckverweisung an die Aussch\u00fcsse in 2./3. Beratung", 
    #                "abstract", "aktivitaet_anzahl", "aktivitaet_anzeige", "aktualisiert", "ausfertigungsdatum", 
    #                "beschlussfassung", "datum", "dokumentart", "einleitungstext", "erlaeuterung", "fortsetzung", "fundstelle", 
    #                "fundstelle.anfangsquadrant", "fundstelle.anfangsseite", "fundstelle.anlagen", "fundstelle.datum", 
    #                "fundstelle.dokumentart", "fundstelle.dokumentnummer", "fundstelle.drucksachetyp", "fundstelle.endquadrant", 
    #                "fundstelle.endseite", "fundstelle.frage_nummer", "fundstelle.herausgeber", "fundstelle.id", "fundstelle.pdf_url", 
    #                "fundstelle.top", "fundstelle.top_zusatz", "fundstelle.urheber", "fundstelle.verteildatum", "fundstelle.xml_url", 
    #                "gang", "heftnummer", "id", "jahrgang", "kom", "mitberaten", "nachtrag", "pdf_url", "ratsdok", "ressort", 
    #                "rubrik_nr", "seite", "titel", "typ", "ueberweisung", "urheber", "verkuendungsblatt_bezeichnung", 
    #                "verkuendungsblatt_kuerzel", "verkuendungsdatum", "vorgang_id", "vorgangsposition", "vorgangstyp", 
    #                "zuordnung", "\u00c4nderung der Ausschuss\u00fcberweisung", "\u00c4nderungsantrag", 
    #                "\u00dcberweisung gem\u00e4\u00df \u00a7 80 Abs. 3 Gesch\u00e4ftsordnung BT", 
    #                "\u00dcberweisung gem\u00e4\u00df \u00a7 92 Gesch\u00e4ftsordnung BT", 
    #                "\u00dcberweisung gem\u00e4\u00df \u00a7 93 Gesch\u00e4ftsordnung BT", 
    #                "\u00dcberweisung gem\u00e4\u00df \u00a7 93 Gesch\u00e4ftsordnung BT: \u00c4nderung der Ausschuss\u00fcberweisung"],
    #     "objects": ["Aktuelle Stunde", "Allgemeine Verwaltungsvorschrift", "Anmerkung zur Plenarsitzung", 
    #                 "Ansprache/Erkl\u00e4rung/Mitteilung", "Antrag", "Befragung der Bundesregierung", "Bericht, Gutachten, Programm", 
    #                 "Besetzung externer Gremien durch BR", "Besetzung externer Gremien durch BT", "Besetzung interner Gremien des BT", 
    #                 "Bestellung und Abberufung von MdBR und stellv. MdBR", "EU-Vorlage", "Enquete-Kommission", 
    #                 "Entlastung der Bundesregierung", "Entlastung des Bundesrechnungshofes", "Entschlie\u00dfungsantrag BT", 
    #                 "Gesch\u00e4ftsordnung", "Gesetzgebung", "Gro\u00dfe Anfrage", "Immunit\u00e4tsangelegenheit", "Kleine Anfrage", 
    #                 "M\u00fcndliche Frage", "Parlamentarische Sonderkommission", "Petition", "Rechtsverordnung", 
    #                 "Rechtsverordnung (Au\u00dfenwirtschaftsrecht)", "Regierungserkl\u00e4rung", "Schriftliche Frage", 
    #                 "Selbst\u00e4ndiger Antrag von L\u00e4ndern auf Entschlie\u00dfung", 
    #                 "Unterrichtung durch das Europ\u00e4ische Parlament", "Untersuchungsausschuss", "Vereinbarte Debatte", 
    #                 "Verfahren vor dem Bundesverfassungsgericht", "Verordnungsantrag der L\u00e4nder", 
    #                 "Vertrauensantrag des Bundeskanzlers", "Wahl der Ausschussvorsitzenden BR", 
    #                 "Wahl der Richter des Bundesverfassungsgerichts ", "Wahl der Schriftf\u00fchrer BR", 
    #                 "Wahl des Bundespr\u00e4sidenten", "Wahl des Pr\u00e4sidiums BR", "Wahl im BT", "Wahlperiodenwechsel", 
    #                 "Wahlpr\u00fcfungsverfahren", "abstract", "aktualisiert", "archiv", "beratungsstand", "datum", "deskriptor", 
    #                 "gesta", "id", "initiative", "inkrafttreten", "kom", "mitteilung", "ratsdok", "sachgebiet", "titel", "typ", 
    #                 "verkuendung", "vorgang_verlinkung", "vorgangstyp", "wahlperiode", "zustimmungsbeduerftigkeit", 
    #                 "\u00dcber- und au\u00dferplanm\u00e4\u00dfige Haushaltsausgaben"]
    # },
    
    # Do I really need this? I mean, it's a reasoning benchmark, not a process log. But it can be added later if needed.
    "reasoning_benchmark": {
        "events": ["Abductive Reasoning - IND", "Abductive Reasoning - NE", "Abductive Reasoning - PE", "Backtracking - IND", 
                   "Backtracking - NE", "Backtracking - PE", "Conclusion - C", "Conclusion - PC", "Conclusion - W", 
                   "Counterfactual Reasoning - IND", "Counterfactual Reasoning - NE", "Counterfactual Reasoning - PE", 
                   "Deductive Reasoning - IND", "Deductive Reasoning - NE", "Deductive Reasoning - PE", 
                   "Ethical or Moral Reasoning - IND", "Ethical or Moral Reasoning - PE", "Heuristic Reasoning - IND", 
                   "Heuristic Reasoning - NE", "Heuristic Reasoning - PE", "Hypothesis Generation - IND", "Hypothesis Generation - NE", 
                   "Hypothesis Generation - PE", "Inductive Reasoning - IND", "Inductive Reasoning - NE", "Inductive Reasoning - PE", 
                   "Pattern Recognition - IND", "Pattern Recognition - NE", "Pattern Recognition - PE", "Validation - IND", 
                   "Validation - NE", "Validation - PE", "text"],
        "objects": ["MOD", "MODQUE"]
    },
    "renting": {
        "events": ["Accept Late Payment", "Apply for Viewing Appointment", "Check Paperwork", "Evict Tenant", "Extensive Screening", 
                   "Hand In The Paperwork", "Issue Warning", "Miss Rent Payment", "Move In", "Pay Rent", "Reject Prospective Tenant", 
                   "Screen Prospective Tenant", "Set Appointment", "Sign Contract", "Tenant Cancels Appartment", "View The Property"],
        "objects": ["Person"]
    },
    # I feel like this one is too much noise
    # "blockchain": {
    #     "events": ["accept contribution to crowdsourcer", "burn tokens", "burned tokens", "call and transfer Ether", 
    #                "call for trusted transfer", "call to buy (public)", "call to buy complete sets", 
    #                "call to buy complete sets with cash (public)", "call to cancel order", "call to check if container for fee token", 
    #                "call to check if container for fee window", "call to check if container for reporting participant", 
    #                "call to check if container for share token", "call to check if container market", "call to check if fork happened", 
    #                "call to check if initialization happened", "call to check if known universe", "call to claim trading proceeds", 
    #                "call to create", "call to create categorical market", "call to create dispute crowdsourcer", 
    #                "call to create initial reporter", "call to create mailbox", "call to create map", "call to create market", 
    #                "call to create order", "call to create scalar market", "call to create share token", "call to create yes-no-market", 
    #                "call to decrement open interest", "call to fill best order", "call to fill best order with limit", 
    #                "call to fill order", "call to find bounding orders", "call to get REP token", 
    #                "call to get dispute round duration in seconds", "call to get or cache designated report no show bond", 
    #                "call to get or cache designated reporter stake", "call to get or cache market creation cost", 
    #                "call to get or cache reporting fee divisor", "call to get or cache validity bond", 
    #                "call to get or create fee window before", "call to get or create next fee window", "call to get timestamp", 
    #                "call to increment open interest", "call to initialize", "call to initialize crowdsourcer", 
    #                "call to initiate a trade", "call to log REP token transferred", "call to log complete set purchase", 
    #                "call to log complete sets sold", "call to log dispute crowdsourcer completion", 
    #                "call to log dispute crowdsourcer contribution", "call to log dispute crowdsourcer redemption", 
    #                "call to log dispute crowdsourcer tokens burned", "call to log dispute crowdsourcer tokens minted", 
    #                "call to log fee token burning", "call to log fee token minted", "call to log fee window burned", 
    #                "call to log fee window minted", "call to log fee window redeemed", "call to log initial report submission", 
    #                "call to log market creation", "call to log order cancellation", "call to log order creation", 
    #                "call to log order filling", "call to log reporter redemption", "call to log share token burned", 
    #                "call to log share token minted", "call to log share tokens transfer", "call to log trading proceeds claim", 
    #                "call to notify about dispute crowdsourcer creation", "call to purchase complete sets", "call to sell complete sets", 
    #                "call to sell complete sets (public)", "call to trade with limit", "call without Ether transfer", "cancel order", 
    #                "claim trading proceeds", "complete dispute", "contribute to dispute", "create crowdsourcer", "create dispute", 
    #                "create disputer for crowdsourcer", "create fee window", "create market", "create new accounting instance", 
    #                "create new contract", "create order", "create universe", "delegate call", 
    #                "delegate call for trusted cancel order transfer", "delegate call for trusted fee window transfer", 
    #                "delegate call for trusted fill order transfer", "delegate call for trusted market transfer",
    #                "delegate call for trusted order transfer", "delegate call for trusted reporting participant transfer", 
    #                "delegate call for trusted universe buy", "delegate call for trusted universe transfer", 
    #                "delegate call to add an entry to a map", "delegate call to approve", "delegate call to assert balances", 
    #                "delegate call to assert is not better price", "delegate call to assert is not worse price", 
    #                "delegate call to burn fee window", "delegate call to buy Atto tokens", "delegate call to buy participant tokens", 
    #                "delegate call to check for better price", "delegate call to check for container for share token", 
    #                "delegate call to check if active", "delegate call to check if container for fee token", 
    #                "delegate call to check if container for fee window", "delegate call to check if container for market", 
    #                "delegate call to check if container for reporting participant", 
    #                "delegate call to check if designated reporter showed", "delegate call to check if designated reporter was correct", 
    #                "delegate call to check if fee window is over", "delegate call to check if market is finalized", 
    #                "delegate call to check if universe is forking", "delegate call to check if worse price", 
    #                "delegate call to check validity", "delegate call to contribute", "delegate call to create a categorical market", 
    #                "delegate call to create scalar market", "delegate call to create shares", "delegate call to create yes-no-market", 
    #                "delegate call to decrement open interest", "delegate call to decrement open interest from market", 
    #                "delegate call to decrement total escrow", "delegate call to deposit Ether", 
    #                "delegate call to deposit Ether for Cash token", "delegate call to derive market creator fee amount", 
    #                "delegate call to derive payout distribution hash", "delegate call to destroy shares", 
    #                "delegate call to do initial report", "delegate call to get REP token", "delegate call to get allowance", 
    #                "delegate call to get amount of an order", "delegate call to get as address or zero", 
    #                "delegate call to get balance", "delegate call to get best order ID", "delegate call to get better order ID", 
    #                "delegate call to get crowdsourcer", "delegate call to get current fee window", "delegate call to get decimals", 
    #                "delegate call to get denomination token", "delegate call to get designated reporter", 
    #                "delegate call to get dispute round duration in seconds", "delegate call to get dispute threshold for fork", 
    #                "delegate call to get end time", "delegate call to get fee token", "delegate call to get fee window", 
    #                "delegate call to get finalization time", "delegate call to get forking market", "delegate call to get market", 
    #                "delegate call to get market creator mailbox", "delegate call to get number of designated reporter no-shows", 
    #                "delegate call to get number of incorrect designated report markets", "delegate call to get number of invalid markets", 
    #                "delegate call to get number of markets", "delegate call to get number of participants", 
    #                "delegate call to get number of ticks", "delegate call to get or cache designated reporter no-show-bond", 
    #                "delegate call to get or cache designated reporter stake", "delegate call to get or cache market creation cost", 
    #                "delegate call to get or cache next fee window", "delegate call to get or cache reporting fee divisor", 
    #                "delegate call to get or cache validity bond", "delegate call to get or create fee window before", 
    #                "delegate call to get order ID", "delegate call to get order creator", "delegate call to get order money escrowed", 
    #                "delegate call to get order shares escrowed", "delegate call to get order type", "delegate call to get outcome", 
    #                "delegate call to get payout distribution hash", "delegate call to get payout numerator", 
    #                "delegate call to get price", "delegate call to get report time stamp", "delegate call to get reporting participant", 
    #                "delegate call to get share token", "delegate call to get size", "delegate call to get stake", 
    #                "delegate call to get start time", "delegate call to get the number of outcomes", 
    #                "delegate call to get the total escrow", "delegate call to get total supply", "delegate call to get universe", 
    #                "delegate call to get winning payout distribution hash", "delegate call to get winning payout numerator", 
    #                "delegate call to get worse order ID", "delegate call to increment open interest", 
    #                "delegate call to increment the total escrowed", "delegate call to initialize", "delegate call to liquidate losing", 
    #                "delegate call to market", "delegate call to migrate", "delegate call to mint fee tokens", 
    #                "delegate call to mint for reporting participant", "delegate call to record fill order", 
    #                "delegate call to redeem", "delegate call to redeem for reporting participant", "delegate call to remove order", 
    #                "delegate call to report", "delegate call to save order", "delegate call to set price", 
    #                "delegate call to transfer token", "delegate call to transfer token on behalf of the owner", 
    #                "delegate call to update statistics once market is finalized", "delegate call to withdraw Ether", 
    #                "delegate call to withdraw Ether (2)", "delegate call to withdraw Ether if possible", "fill order", 
    #                "finalize crowdsourcer", "finalize market", "give approval to transfer tokens", "initialize crowdsourcer", 
    #                "mint tokens", "minted tokens", "purchase complete sets", "redeem as initial reporter", "redeem dispute crowdsourcer", 
    #                "redeem fee window", "sell complete sets", "submit initial report", "transfer market", "transfer tokens", 
    #                "transferred tokens", "undecoded event", "withdraw contribution to crowdsourcer", "withdraw fees from crowdsourcer", 
    #                "withdraw proceeds from crowdsourcer"],
    #     "objects": ["CONTRACT", "EOA", "MARKET", "ORDER", "TOKEN"]
    # },
    "lending": {
        "events": ["Application Rejected", "Appointment Denied", "Hand In Credit Appliaction", "Loan Denied", 
                   "Make Visit to Assess Colatteral", "Request Appointment", "Request Co-Signer On Loan", "Set Appointment", 
                   "Sign Loan Agreement", "Submit File to Underwriter", "Verify Borrowers Information", "skipped_examination"],
        "objects": ["Age", "Case", "Creditscore", "Person", "Yearsofeducation", "isCitizen", "isGender", 
                    "isGermanspeaking", "numOcc_ApplicationRejected", "numOcc_AppointmentDenied", "numOcc_HandInCreditAppliaction", 
                    "numOcc_LoanDenied", "numOcc_MakeVisittoAssessColatteral", "numOcc_RequestAppointment", 
                    "numOcc_RequestCo-SignerOnLoan", "numOcc_SetAppointment", "numOcc_SignLoanAgreement", "numOcc_SubmitFiletoUnderwriter", 
                    "numOcc_VerifyBorrowersInformation", "numOcc_skipped_examination", "overallProtected", "reworkedActivities", 
                    "sumBoolDiscrFactors", "throughputTime"]
    },
    # "01_o2c": {
    #     "events": ["Apply Discount", "Apply Payment", "Approve Order", "Approve Return", "Generate Invoice", "Generate Pick List", "Generate Shipping Label", "Handle Payment Discrepancy", "Issue Credit Note", "Issue Payment Reminder", "Pack Items", "Perform Credit Check", "Pick Items", "Process Return Request", "Receive Customer Order", "Receive Payment", "Reject Order", "Schedule Order Fulfillment", "Send Invoice to Customer", "Ship Order", "Update Inventory", "Validate Customer Order"],
    #     "objects": ["CreditNote", "Customer", "Dispute", "Invoice", "Order", "Payment", "Product", "Return", "Shipment"]
    # },
    # "04_hospital": {
    #     "events": ["Appointment Scheduling", "Assign Bed", "Diagnostic Test Order", "Initial Evaluation", "Insurance Verification", "Patient Arrival", "Perform Diagnostic Test", "Pre-Admission Consultation", "Registration"],
    #     "objects": ["Bed", "DiagnosticEquipment", "InsuranceCompany", "LabTechnician", "Nurse", "Patient", "Physician", "Room"]
    # },
    "container_logistics": {
        "events": ["Book Vehicles", "Bring to Loading Bay", "Collect Goods", "Create Transport Document", "Depart", "Drive to Terminal", 
                   "Load Truck", "Load to Vehicle", "Order Empty Containers", "Pick Up Empty Container", "Place in Stock", 
                   "Register Customer Order", "Reschedule Container", "Weigh"],
        "objects": ["AmountofContainers", "AmountofGoods", "AmountofHandlingUnits", "Container", "Customer Order", 
                    "DepartureDate", "Forklift", "Handling Unit", "Status", "Transport Document", "Truck", 
                    "Vehicle", "Weight"]
    },
}


In [3]:
def generate_example():
    pattern = random.choice(patterns)
    
    # Pick a domain first to ensure semantic consistency
    domain = random.choice(list(domains.keys()))
    domain_data = domains[domain]

    A = random.choice(domain_data["events"])
    # Ensure event A and B are different for better logical examples
    B = random.choice([e for e in domain_data["events"] if e != A])
    obj = random.choice(domain_data["objects"])
    t = random.choice([1, 6, 24, 72])

    text = pattern.replace("EVENT_A", A)\
                  .replace("EVENT_B", B)\
                  .replace("OBJECT", obj)\
                  .replace("TIME", str(t))

    # Dynamically inject constraints based on the pattern
    constraints = []
    filters = []
    if "before" in pattern or "eventually" in pattern:
        constraints = [{
            "type": "Filter",
            "filter": { "type": "TimeBetweenEvents", "from_event": 0, "to_event": 1, "min_seconds": 0, "max_seconds": None }
        }]
    elif "within TIME hours" in pattern:
        constraints = [{
            "type": "Filter",
            "filter": { "type": "TimeBetweenEvents", "from_event": 0, "to_event": 1, "min_seconds": 0, "max_seconds": t * 3600 }
        }]
    elif "must not occur" in pattern:
        # Requires a child edge with a NOT constraint to be completely valid OCPQ
        constraints = [{
            "type": "NOT",
            "child_names": ["edge_1"]
        }]
    elif "refer to the same OBJECT" in pattern:
        filters = [
            { "type": "O2E", "object": 0, "event": 0, "qualifier": None },
            { "type": "O2E", "object": 0, "event": 1, "qualifier": None }
        ]

    ocpq_tree = {
            "nodes": [
                {
                    "Box": [
                        {
                            "newEventVars": { "0": [A], "1": [B] },
                            "newObjectVars": { "0": [obj] },
                            "filters": filters,
                            "sizeFilters": [],
                            "constraints": constraints,
                            "evVarLabels": {},
                            "obVarLabels": {}
                        },
                        [] # IDs of any connected child nodes in the tree
                    ]
                }
            ],
            "edgeNames": []
    }

    return {"nl": text, "query": ocpq_tree}

dataset = []
seen = set()

while len(dataset) < 200:
    example = generate_example()
    # Use the natural language text to check for uniqueness
    if example["nl"] not in seen:
        seen.add(example["nl"])
        dataset.append(example)

with open("ocpq_dataset.json","w") as f:
    json.dump(dataset,f,indent=2)

In [4]:
dataset[:5]

[{'nl': 'review application and submit application refer to the same candidates',
  'query': {'nodes': [{'Box': [{'newEventVars': {'0': ['review application'],
        '1': ['submit application']},
       'newObjectVars': {'0': ['candidates']},
       'filters': [{'type': 'O2E', 'object': 0, 'event': 0, 'qualifier': None},
        {'type': 'O2E', 'object': 0, 'event': 1, 'qualifier': None}],
       'sizeFilters': [],
       'constraints': [],
       'evVarLabels': {},
       'obVarLabels': {}},
      []]}],
   'edgeNames': []}},
 {'nl': 'After ship package, add item must not occur',
  'query': {'nodes': [{'Box': [{'newEventVars': {'0': ['ship package'],
        '1': ['add item']},
       'newObjectVars': {'0': ['items']},
       'filters': [],
       'sizeFilters': [],
       'constraints': [{'type': 'NOT', 'child_names': ['edge_1']}],
       'evVarLabels': {},
       'obVarLabels': {}},
      []]}],
   'edgeNames': []}},
 {'nl': 'create order and cancel order refer to the same items',